In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss

import warnings
warnings.filterwarnings("ignore")

# 1. Load Data


In [2]:
train_path = "Train.csv"
test_path = "Test.csv"
sample_path = "SampleSubmission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_df = pd.read_csv(sample_path)

# 2. Preprocess


In [3]:
# Lowercase text (force string)
train_df["text"] = train_df["text"].astype(str).str.lower()
test_df["text"] = test_df["text"].astype(str).str.lower()


# Encode labels safely
label_encoder = LabelEncoder()

train_df["label_enc"] = pd.Series(
    label_encoder.fit_transform(train_df["label"]),
    index=train_df.index
)

print("Classes:", label_encoder.classes_)


X = train_df["text"]
y = train_df["label_enc"]

Classes: ['Alcohol' 'Depression' 'Drugs' 'Suicide']


# 3. Train / Validation Split


In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=42
)


# 4. Build Pipeline


In [5]:
pipeline = Pipeline([

    # Text → TF-IDF vectors
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50000,
        min_df=2,
        stop_words="english"
    )),

    # Linear SVM
    ("svc", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=42
    ))
])

# 5. Calibrate for Probabilities


In [6]:
model = CalibratedClassifierCV(
    pipeline,
    method="sigmoid",
    cv=5
)


# 6. Train


In [7]:
print("Training model...")

model.fit(X_train, y_train)

print("Training done.")

Training model...
Training done.


# 7. Validate 


In [8]:
val_probs = model.predict_proba(X_val)

val_loss = log_loss(y_val, val_probs)

print("Validation Log Loss:", val_loss)


Validation Log Loss: 0.5504536797138098


# 8. Predict Test Set


In [10]:
print("Predicting test set...")

# convert to ndarray so predict_proba sees a MatrixLike object
test_texts = test_df["text"].to_numpy()
test_probs = model.predict_proba(test_texts)

Predicting test set...


# 9. Create Submission


In [13]:
# Get correct class order
class_names = label_encoder.classes_

submission = pd.DataFrame()
submission["ID"] = test_df["ID"]

# Fill using real class order
for i, class_name in enumerate(class_names):
    submission[class_name] = test_probs[:, i]

# Reorder to competition format
submission = submission[[
    "ID",
    "Depression",
    "Alcohol",
    "Suicide",
    "Drugs"
]]

submission.to_csv("submission_svc.csv", index=False)

print("Saved submission_svc.csv")
print(submission.head())

Saved submission_svc.csv
         ID  Depression   Alcohol   Suicide     Drugs
0  02V56KMO    0.567734  0.169595  0.231598  0.031073
1  03BMGTOK    0.874090  0.036808  0.063234  0.025868
2  03LZVFM6    0.949183  0.011166  0.030171  0.009481
3  0EPULUM5    0.830499  0.021605  0.128619  0.019277
4  0GM4C5GD    0.111167  0.237212  0.069320  0.582301


# 10. Visualisation


In [14]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

In [16]:
# Extract trained pipeline from calibrated model
calibrated_model = model.calibrated_classifiers_[0]

pipeline = calibrated_model.estimator

tfidf = pipeline.named_steps["tfidf"]
svc = pipeline.named_steps["svc"]

X_vec = tfidf.transform(X)